<a href="https://colab.research.google.com/github/sparshbansal-newton/deep-learning-labs/blob/main/Notebooks/5_Loss_Functions/Loss_Functions_LAB4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Loss Functions


Loss functions tell a model **how wrong it is** — a single number to push down during training.

In this lab we build 3 losses by hand and check them against PyTorch:

| Part | Task | Loss | Data |
|---|---|---|---|
| 1 | Regression | **MSE** vs **MAE** | California Housing 🏡 |
| 2 | Binary classification | **BCE** | Breast Cancer 🩺 |
| 3 | Multi-class | **Cross-Entropy** | Handwritten Digits ✍️ |

> 💡 **Big idea:** the *right* loss depends on the task — and feeding it the *right input* (probabilities vs raw logits) matters just as much.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing, load_breast_cancer, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
torch.manual_seed(42)

## Part 1 · Regression — predicting house prices 🏡

We predict median house value from one feature, **median income** (`MedInc`).

- **MSE** (Mean Squared Error) → squares each error, so **big mistakes hurt a lot**.
- **MAE** (Mean Absolute Error) → absolute error, so **every mistake counts equally**.

We'll also inject a few **outliers** to see which loss stays calm. 👀

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame
df.head()

In [ ]:
# 👀 A quick look at the full dataset: 8 features + target `MedHouseVal`
print("Rows, cols:", df.shape)
df.describe().round(2)

In [ ]:
df_sub = df.sample(n=400, random_state=42).reset_index(drop=True)

print("Rows, cols:", df_sub.shape)

> 💡 We keep it tiny on purpose: **400 rows, 1 feature** (`MedInc`). Small + 1-D means we can *plot* the loss later and literally see what each loss does.

In [ ]:
X_reg = df_sub[["MedInc"]].values.astype(np.float32)
y_reg = df_sub["MedHouseVal"].values.astype(np.float32)

In [ ]:
outlier_idx = [5, 120, 300]
y_reg_outliers = y_reg.copy()
y_reg_outliers[outlier_idx] += 20.0

In [ ]:
# 🧨 We bumped 3 house prices up by +20 to act as outliers.
# Compare the original vs corrupted target for those exact rows:
import pandas as pd
pd.DataFrame({
    "MedInc":         X_reg[outlier_idx, 0],
    "y_clean":        y_reg[outlier_idx],
    "y_with_outlier": y_reg_outliers[outlier_idx],
})

In [ ]:
y_reg.shape

In [ ]:
X_reg_t = torch.tensor(X_reg)
y_clean_t = torch.tensor(y_reg)
y_outlier_t = torch.tensor(y_reg_outliers)

### A fresh, untrained model
`nn.Linear(1, 1)` is just `y = w·x + b` with **random** `w` and `b`. Its predictions are bad on purpose — that's exactly what gives the loss something to measure.

In [ ]:
torch.manual_seed(42)
model = nn.Linear(in_features=1, out_features=1)
model.weight.item()

In [ ]:
model.bias.item()

In [ ]:
y_pred_t = model(X_reg_t)
y_pred_t = y_pred_t.squeeze(1)
print("First 5 predictions:", y_pred_t[:5])

### MSE by hand

$$\text{MSE} = \frac{1}{N}\sum \left(y_{true} - y_{pred}\right)^2$$

Square the errors, then average. The square is why large errors dominate.

In [ ]:
def mse_manual(y_true, y_pred):
    result= torch.mean((y_true - y_pred) ** 2)
    return result

In [ ]:
mse_clean_manual = mse_manual(y_clean_t, y_pred_t)
mse_outliers_manual = mse_manual(y_outlier_t, y_pred_t)
print(f"Manual Clean data   -> MSE: {mse_clean_manual.item():.4f}")
print(f"Manual Outlier data -> MSE: {mse_outliers_manual.item():.4f}")

> ✅ **Sanity check:** your manual value should match `nn.MSELoss()` exactly.
> Notice how outliers **inflate MSE** (4.48 → 6.98) — squaring blows the 3 big errors way up.

In [ ]:
mse_loss_fn = nn.MSELoss()
mse_clean = mse_loss_fn(y_pred_t, y_clean_t)
mse_outliers = mse_loss_fn(y_pred_t, y_outlier_t)
print(f" Built-in Clean data   -> MSE: {mse_clean.item():.4f}")
print(f" Built-in Outlier data -> MSE: {mse_outliers.item():.4f}")

### 🔧 Your turn: implement MAE

$$\text{MAE} = \frac{1}{N}\sum \left| y_{true} - y_{pred} \right|$$

Fill in `mae_manual` in the next cell, then run it on both target sets.

> 💡 **Hints**
> - Absolute value: `torch.abs(...)`  (or `(...).abs()`)
> - Average with `torch.mean(...)` — it's the same shape as `mse_manual`, just swap `**2` for `abs`.
> - Return a **tensor** (not a plain float) so the `torch.allclose` check passes.
> - Expected result: **Match? True**, and MAE rises *far less* than MSE did under outliers.

<details><summary>👀 Stuck? click for the solution</summary>

```python
def mae_manual(y_true, y_pred):
    return torch.mean(torch.abs(y_true - y_pred))

mae_clean_manual    = mae_manual(y_clean_t, y_pred_t)
mae_outliers_manual = mae_manual(y_outlier_t, y_pred_t)
```
</details>

In [ ]:
# 🔧 Your turn — implement Mean Absolute Error manually,
# then compute it for BOTH the clean and outlier targets.

def mae_manual(y_true, y_pred):
    # TODO: replace None with your implementation
    result = None
    return result

# TODO: use your function on both target sets
mae_clean_manual = None
mae_outliers_manual = None

mae_loss_fn = nn.L1Loss()
mae_clean_builtin = mae_loss_fn(y_pred_t, y_clean_t)
mae_outliers_builtin = mae_loss_fn(y_pred_t, y_outlier_t)

if mae_clean_manual is None or mae_outliers_manual is None:
    print("⚠️  Still returning None — fill in mae_manual above.")
else:
    print(f"Clean    -> Your MAE: {mae_clean_manual.item():.4f}   | Built-in: {mae_clean_builtin.item():.4f}")
    print(f"Outliers -> Your MAE: {mae_outliers_manual.item():.4f}   | Built-in: {mae_outliers_builtin.item():.4f}")
    print("Match?",
          torch.allclose(mae_clean_manual, mae_clean_builtin, atol=1e-5) and
          torch.allclose(mae_outliers_manual, mae_outliers_builtin, atol=1e-5))


### 📈 Why it matters: the loss landscape
We sweep the weight `w` and plot the loss. Compare **blue** (clean) vs **red** (with outliers):

- **MSE** → outliers *drag* the lowest point sideways, so the model chases them.
- **MAE** → the minimum barely moves — it's **robust** to outliers.

> 🎯 **Takeaway:** clean data → MSE is fine. Data with outliers → MAE is safer.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

weights = np.linspace(-1, 1, 100)
bias_fixed = 0.0

def sweep_loss(loss_fn, X, y, weights, bias):
    losses = []
    for w in weights:
        preds = w * X.squeeze() + bias  # [N]
        preds_t = torch.tensor(preds, dtype=torch.float32)
        losses.append(loss_fn(preds_t, torch.tensor(y, dtype=torch.float32)).item())
    return np.array(losses)

mse_sweep_clean = sweep_loss(nn.MSELoss(), X_reg, y_reg, weights, bias_fixed)
mse_sweep_out = sweep_loss(nn.MSELoss(), X_reg, y_reg_outliers, weights, bias_fixed)
mae_sweep_clean = sweep_loss(nn.L1Loss(), X_reg, y_reg, weights, bias_fixed)
mae_sweep_out = sweep_loss(nn.L1Loss(), X_reg, y_reg_outliers, weights, bias_fixed)

fig = make_subplots(rows=1, cols=2, subplot_titles=("MSE loss landscape", "MAE loss landscape"))

fig.add_trace(go.Scatter(x=weights, y=mse_sweep_clean, mode="lines", name="clean",
                          legendgroup="clean", line=dict(color="blue")), row=1, col=1)
fig.add_trace(go.Scatter(x=weights, y=mse_sweep_out, mode="lines", name="with outliers",
                          legendgroup="outliers", line=dict(color="red")), row=1, col=1)

fig.add_trace(go.Scatter(x=weights, y=mae_sweep_clean, mode="lines", name="clean",
                          legendgroup="clean", line=dict(color="blue"), showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=weights, y=mae_sweep_out, mode="lines", name="with outliers",
                          legendgroup="outliers", line=dict(color="red"), showlegend=False), row=1, col=2)

fig.update_xaxes(title_text="weight", row=1, col=1)
fig.update_yaxes(title_text="loss", row=1, col=1)
fig.update_xaxes(title_text="weight", row=1, col=2)
fig.update_yaxes(title_text="loss", row=1, col=2)

fig.update_layout(height=400, width=900,
                   title_text="Loss landscape: MSE vs MAE, clean vs outliers")
fig.show()

## Part 2 · Binary classification — is a tumour benign? 🩺

The target is now **0 or 1**. We use **Binary Cross-Entropy (BCE)**: it rewards confident-correct predictions and heavily punishes confident-wrong ones.

PyTorch has two flavours — and mixing them up is a classic bug 🐞:
- `nn.BCELoss` → feed it **probabilities** (after sigmoid, range 0–1).
- `nn.BCEWithLogitsLoss` → feed it **raw logits** (it runs sigmoid *for you*).

In [ ]:
bc = load_breast_cancer()
X_bc = bc.data.astype(np.float32)
y_bc = bc.target.astype(np.float32)

In [ ]:
print("X_bc shape:", X_bc.shape)
print("y_bc shape:", y_bc.shape)
print("Class balance (0=malignant, 1=benign):", np.bincount(y_bc.astype(int)))

In [ ]:
# 👀 The 30 features as a dataframe. target: 0 = malignant, 1 = benign
import pandas as pd
bc_df = pd.DataFrame(bc.data, columns=bc.feature_names)
bc_df["target"] = bc.target
bc_df.head()

In [ ]:
X_train_bc, X_test_bc, y_train_bc, y_test_bc = train_test_split(
    X_bc, y_bc, test_size=0.2, random_state=42, stratify=y_bc
)

In [ ]:
scaler_bc = StandardScaler()
X_train_bc = scaler_bc.fit_transform(X_train_bc).astype(np.float32)
X_test_bc = scaler_bc.transform(X_test_bc).astype(np.float32)

In [ ]:
X_train_bc_t = torch.tensor(X_train_bc)
y_train_bc_t = torch.tensor(y_train_bc)
X_test_bc_t = torch.tensor(X_test_bc)
y_test_bc_t = torch.tensor(y_test_bc)

In [ ]:
torch.manual_seed(42)
model_bc = nn.Linear(in_features=30, out_features=1)

### Logits → probabilities
The model outputs **logits** (any real number). `sigmoid` squashes each one into a **0–1 probability**.

In [ ]:
# Get raw logits from the (untrained) model

test_logits = model_bc(X_test_bc_t[:5]).squeeze(1)
test_probs = torch.sigmoid(test_logits)

print("Raw logits shape:  ", test_logits.shape)
print("Raw logits:        ", test_logits)
print("Probabilities:     ", test_probs)

In [ ]:
bce_loss_fn = nn.BCELoss()
bce_result = bce_loss_fn(test_probs, y_test_bc_t[:5])

print(f"BCELoss(probabilities): {bce_result.item():.6f}")

> 🐞 **Watch closely!** The next cell makes a mistake *on purpose*: it passes `test_probs` (already sigmoided) into `BCEWithLogitsLoss`, which then applies sigmoid **again**. See how the two numbers disagree — this is one of the most common PyTorch slip-ups.

In [ ]:
# Now let's try BCEWithLogitsLoss too — a "faster" way that skips computing sigmoid ourselves.
# Naturally, we already HAVE test_probs sitting right there... so let's just reuse it.

bce_logits_loss_fn = nn.BCEWithLogitsLoss()
bce_logits_result = bce_logits_loss_fn(test_probs, y_test_bc_t[:5])

print(f"BCELoss(probabilities):        {bce_result.item():.6f}")
print(f"BCEWithLogitsLoss(test_probs): {bce_logits_result.item():.6f}")

In [ ]:
# Let's investigate. What does BCEWithLogitsLoss actually do internally with its input?
# The docs say: "This loss combines a Sigmoid layer and the BCELoss in one single class."
# ...it applies sigmoid ITSELF. So what did we just feed it?

print("What we passed to BCEWithLogitsLoss:", test_probs)
print("What BCEWithLogitsLoss did to it (sigmoid applied AGAIN):", torch.sigmoid(test_probs))
print("\nCompare to what we actually wanted it to see:")
print("sigmoid(test_logits) — the correct probabilities:", torch.sigmoid(test_logits))

> ✅ **Rule of thumb:** `BCEWithLogitsLoss(logits)` — never pass probabilities into it. It's both numerically safer *and* faster than doing `sigmoid` + `BCELoss` yourself.

### 🔧 Your turn: implement BCE by hand

Binary Cross-Entropy, from **probabilities** `p` and labels `y`:

$$\text{BCE} = -\frac{1}{N}\sum \big[\, y\log(p) + (1-y)\log(1-p) \,\big]$$

Fill in `bce_manual` below, then compare to `nn.BCELoss`.

> 💡 **Hints**
> - `p` is `test_probs` (already sigmoided); `y` is `y_test_bc_t[:5]`.
> - Use `torch.log` and average with `torch.mean`.
> - **Clamp** `p` away from 0 and 1 (`torch.clamp(p, eps, 1 - eps)`) so `log` never sees 0.
> - Expected: **Match? True**.

<details><summary>👀 Stuck? click for the solution</summary>

```python
def bce_manual(y_true, p, eps=1e-7):
    p = torch.clamp(p, eps, 1 - eps)
    return -torch.mean(y_true * torch.log(p) + (1 - y_true) * torch.log(1 - p))

bce_manual_result = bce_manual(y_test_bc_t[:5], test_probs)
```
</details>

In [ ]:
# 🔧 Your turn — implement Binary Cross-Entropy from probabilities.

def bce_manual(y_true, p):
    # TODO: replace None with your implementation (remember to clamp p away from 0 and 1)
    result = None
    return result

# TODO: use your function  (p = test_probs, y = y_test_bc_t[:5])
bce_manual_result = None

bce_builtin_result = nn.BCELoss()(test_probs, y_test_bc_t[:5])

if bce_manual_result is None:
    print("⚠️  Still returning None — fill in bce_manual above.")
else:
    print(f"Your BCE: {bce_manual_result.item():.6f}   | Built-in: {bce_builtin_result.item():.6f}")
    print("Match?", torch.allclose(bce_manual_result, bce_builtin_result, atol=1e-5))

## Part 3 · Multi-class — which digit is it? ✍️

Now there are **10 classes** (digits 0–9). We use **Cross-Entropy Loss**, the multi-class cousin of BCE.

> 💡 `nn.CrossEntropyLoss` wants **raw logits** (shape `[N, 10]`) and **integer class labels** — it applies softmax + log internally. Don't softmax first!

In [ ]:
digits = load_digits()
X_dig = digits.data.astype(np.float32)
y_dig = digits.target.astype(np.int64)

print("X_dig shape:", X_dig.shape)
print("y_dig shape:", y_dig.shape)

digits.data[0]

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(10, 2))
for i, ax in enumerate(axes):
    ax.imshow(digits.images[i], cmap="gray")
    ax.set_title(str(digits.target[i]))
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# train test spli
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_dig, y_dig, test_size=0.2, random_state=42, stratify=y_dig
)

scaler_d = StandardScaler()
X_train_d = scaler_d.fit_transform(X_train_d).astype(np.float32)
X_test_d = scaler_d.transform(X_test_d).astype(np.float32)

X_train_d_t = torch.tensor(X_train_d)  # [N_train, 64]
y_train_d_t = torch.tensor(y_train_d)  # [N_train] int64 class ids
X_test_d_t = torch.tensor(X_test_d)    # [N_test, 64]
y_test_d_t = torch.tensor(y_test_d)    # [N_test]

In [ ]:
torch.manual_seed(42)
model_dig = nn.Linear(in_features=64, out_features=10)

print(model_dig.weight.shape)
print(model_dig.bias.shape)

In [ ]:
test_logits_d = model_dig(X_test_d_t[:5])

print("Logits shape:", test_logits_d.shape)
print("Logits for first test image:", test_logits_d[0])

### Cross-Entropy in action
Each row of logits is 10 scores. Cross-Entropy softmaxes them into probabilities, then checks the probability assigned to the **correct** class.

In [ ]:
# CrossEntropyLoss expects raw logits directly (it applies softmax + log internally)
criterion_ce = nn.CrossEntropyLoss()
ce_result = criterion_ce(test_logits_d, y_test_d_t[:5])

print(f"CrossEntropyLoss(raw logits): {ce_result.item():.4f}")

> 👀 Below we peek at softmax ourselves — purely to *see* the probabilities. For the actual loss, always hand `CrossEntropyLoss` the **raw logits**, not these.

In [ ]:
# We know softmax turns logits into class probabilities.
# Since we already have test_logits_d, let's compute probabilities ourselves...
test_probs_d = torch.softmax(test_logits_d, dim=1)

print("Probabilities for first test image:", test_probs_d[0])
print("Do they sum to 1?", test_probs_d[0].sum().item())

### 🔧 Your turn: implement Cross-Entropy by hand

Cross-Entropy = softmax → take the probability of the **correct** class → `-log` → average:

$$\text{CE} = -\frac{1}{N}\sum \log\big(p_{\text{correct class}}\big)$$

Fill in `ce_manual` below, then compare to `nn.CrossEntropyLoss`.

> 💡 **Hints**
> - Start from raw logits: `torch.softmax(logits, dim=1)`.
> - Pick each row's true-class probability: `probs[torch.arange(len(y)), y]`.
> - Take `-torch.log(...)`, then `torch.mean(...)`.
> - Check against `nn.CrossEntropyLoss()(test_logits_d, y_test_d_t[:5])`.

<details><summary>👀 Stuck? click for the solution</summary>

```python
def ce_manual(logits, y):
    probs = torch.softmax(logits, dim=1)
    correct = probs[torch.arange(len(y)), y]
    return -torch.mean(torch.log(correct))

ce_manual_result = ce_manual(test_logits_d, y_test_d_t[:5])
```
</details>

In [ ]:
# 🔧 Your turn — implement Cross-Entropy from raw logits + integer labels.

def ce_manual(logits, y):
    # TODO: replace None with your implementation (softmax -> pick true class -> -log -> mean)
    result = None
    return result

# TODO: use your function on the first 5 test digits
ce_manual_result = None

ce_builtin_result = nn.CrossEntropyLoss()(test_logits_d, y_test_d_t[:5])

if ce_manual_result is None:
    print("⚠️  Still returning None — fill in ce_manual above.")
else:
    print(f"Your CE: {ce_manual_result.item():.4f}   | Built-in: {ce_builtin_result.item():.4f}")
    print("Match?", torch.allclose(ce_manual_result, ce_builtin_result, atol=1e-5))

## 🎁 Recap

| Task | Loss | Feed it | Watch out for |
|---|---|---|---|
| Regression | MSE / MAE | predictions | MSE reacts to outliers, MAE shrugs |
| Binary | `BCEWithLogitsLoss` | **logits** | don't pass probabilities |
| Multi-class | `CrossEntropyLoss` | **logits + int labels** | don't softmax first |

> 🎯 **One line to remember:** choose the loss that fits the task, and feed it **raw logits** whenever the name says *"WithLogits"* or it's `CrossEntropyLoss`.